In [ ]:
from flask import Flask, request, jsonify
from pymongo import MongoClient
from bson.objectid import ObjectId
from werkzeug.serving import run_simple
from flask import render_template

app = Flask(__name__)

# MongoDB Connection
client = MongoClient("mongodb://localhost:27017/")
db = client["inventory"]

customers_collection = db["customers"]
products_collection = db["products"]
orders_collection = db["order"]

# ----------------------
# Helper Function
# ----------------------
def serialize(doc):
    doc["_id"] = str(doc["_id"])
    return doc

@app.route('/')
def home():
    return render_template('index.html')
# ======================
# PRODUCTS CRUD
# ======================

@app.route('/products', methods=['POST'])
def create_product():
    data = request.json
    result = products_collection.insert_one(data)
    return jsonify({"message": "Product created", "id": str(result.inserted_id)})

@app.route('/products', methods=['GET'])
def get_products():
    products = [serialize(p) for p in products_collection.find()]
    return jsonify(products)

@app.route('/products/<id>', methods=['GET'])
def get_product(id):
    product = products_collection.find_one({"_id": ObjectId(id)})
    if product:
        return jsonify(serialize(product))
    return jsonify({"error": "Product not found"}), 404

@app.route('/products/<id>', methods=['PUT'])
def update_product(id):
    data = request.json
    products_collection.update_one({"_id": ObjectId(id)}, {"$set": data})
    return jsonify({"message": "Product updated"})

@app.route('/products/<id>', methods=['DELETE'])
def delete_product(id):
    products_collection.delete_one({"_id": ObjectId(id)})
    return jsonify({"message": "Product deleted"})

# ======================
# CUSTOMERS CRUD
# ======================

@app.route('/customers', methods=['POST'])
def create_customer():
    data = request.json
    result = customers_collection.insert_one(data)
    return jsonify({"message": "Customer created", "id": str(result.inserted_id)})

@app.route('/customers', methods=['GET'])
def get_customers():
    customers = [serialize(c) for c in customers_collection.find()]
    return jsonify(customers)

@app.route('/customers/<id>', methods=['GET'])
def get_customer(id):
    customer = customers_collection.find_one({"_id": ObjectId(id)})
    if customer:
        return jsonify(serialize(customer))
    return jsonify({"error": "Customer not found"}), 404

@app.route('/customers/<id>', methods=['PUT'])
def update_customer(id):
    data = request.json
    customers_collection.update_one({"_id": ObjectId(id)}, {"$set": data})
    return jsonify({"message": "Customer updated"})

@app.route('/customers/<id>', methods=['DELETE'])
def delete_customer(id):
    customers_collection.delete_one({"_id": ObjectId(id)})
    return jsonify({"message": "Customer deleted"})

# ======================
# ORDERS CRUD
# ======================

@app.route('/order', methods=['POST'])
def create_order():
    data = request.json

    total_price = 0
    for item in data.get("products", []):
        total_price += item["quantity"] * item["price"]

    data["total_price"] = total_price

    result = orders_collection.insert_one(data)
    return jsonify({"message": "Order created", "id": str(result.inserted_id)})

@app.route('/order', methods=['GET'])
def get_orders():
    orders = []
    for o in orders_collection.find():
        order = serialize(o)

        # Ensure products list is clear and structured
        formatted_products = []
        for item in order.get("products", []):
            formatted_products.append({
                "product_name": item.get("product_name"),
                "quantity": item.get("quantity"),
                "price": item.get("price"),
                "subtotal": item.get("quantity", 0) * item.get("price", 0)
            })

        order["products"] = formatted_products
        orders.append(order)

    return jsonify(orders)

@app.route('/order/<id>', methods=['GET'])
def get_order(id):
    order = orders_collection.find_one({"_id": ObjectId(id)})
    if order:
        return jsonify(serialize(order))
    return jsonify({"error": "Order not found"}), 404

@app.route('/order/<id>', methods=['PUT'])
def update_order(id):
    data = request.json

    # Remove _id from the data if it exists so MongoDB doesn't try to update it
    if "_id" in data:
        del data["_id"]

    if "products" in data:
        total_price = 0
        for item in data["products"]:
            # Logic check: ensures quantity and price exist to avoid key errors
            qty = item.get("quantity", 0)
            price = item.get("price", 0)
            total_price += qty * price
        data["total_price"] = total_price

    orders_collection.update_one({"_id": ObjectId(id)}, {"$set": data})
    
    # Return the updated object or the ID to keep the frontend happy
    return jsonify({"message": "Order updated", "_id": id})

# ======================
# RUN SERVER
# ======================

if __name__ == '__main__':
    run_simple("localhost", 5000, app) # Run the Flask app in a notebook


 * Running on http://localhost:5000
Press CTRL+C to quit
127.0.0.1 - - [28/Apr/2026 13:15:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [28/Apr/2026 13:15:28] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [28/Apr/2026 13:16:09] "GET /products HTTP/1.1" 200 -
